# Student Performance Analysis
**Tools:** PySpark | Jupyter Notebook | Anaconda  
**Objective:** Load CSV → Clean Data → Avg Marks per Dept → Top 5 Students

## Step 1: Create Sample Dataset (CSV)

In [9]:
import csv

data = [
    ['ID', 'Name', 'Department', 'Marks'],
    [1, 'Ali Hassan',       'Computer Science', 88],
    [2, 'Sara Khan',        'Mathematics',       92],
    [3, 'Usman Tariq',      'Computer Science', 76],
    [4, 'Ayesha Noor',      'Physics',           85],
    [5, 'Bilal Ahmed',      'Mathematics',       90],
    [6, 'Fatima Malik',     'Physics',           78],
    [7, 'Hamza Raza',       'Computer Science', 95],
    [8, 'Zara Iqbal',       'Mathematics',       '',],   # missing value
    [9, 'Omar Farooq',      'Physics',           82],
    [10,'Nadia Siddiqui',   'Computer Science', 91],
    [11,'Kamran Baig',      'Mathematics',       74],
    [12,'Hina Butt',        'Physics',           '',],   # missing value
    [13,'Tariq Mehmood',    'Computer Science', 69],
    [14,'Sana Javed',       'Mathematics',       88],
    [15,'Rizwan Shah',      'Physics',           93],
]

with open('students.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(data)

print('students.csv created successfully!')

students.csv created successfully!


## Step 2: Start PySpark Session

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, desc, round as spark_round
from pyspark.sql.types import IntegerType


spark = SparkSession.builder \
    .appName('StudentPerformanceAnalysis') \
    .getOrCreate()

print('Spark Session Started!')
print(f'   Spark Version: {spark.version}')

Spark Session Started!
   Spark Version: 4.0.2


## Step 3: Load CSV Dataset

In [11]:

df = spark.read.csv('students.csv', header=True, inferSchema=True)

print(' Dataset Loaded Successfully!')
print(f'   Total Rows    : {df.count()}')
print(f'   Total Columns : {len(df.columns)}')
print()
print(' Schema:')
df.printSchema()

print(' Raw Data Preview:')
df.show()

 Dataset Loaded Successfully!
   Total Rows    : 15
   Total Columns : 4

 Schema:
root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Marks: integer (nullable = true)

 Raw Data Preview:
+---+--------------+----------------+-----+
| ID|          Name|      Department|Marks|
+---+--------------+----------------+-----+
|  1|    Ali Hassan|Computer Science|   88|
|  2|     Sara Khan|     Mathematics|   92|
|  3|   Usman Tariq|Computer Science|   76|
|  4|   Ayesha Noor|         Physics|   85|
|  5|   Bilal Ahmed|     Mathematics|   90|
|  6|  Fatima Malik|         Physics|   78|
|  7|    Hamza Raza|Computer Science|   95|
|  8|    Zara Iqbal|     Mathematics| NULL|
|  9|   Omar Farooq|         Physics|   82|
| 10|Nadia Siddiqui|Computer Science|   91|
| 11|   Kamran Baig|     Mathematics|   74|
| 12|     Hina Butt|         Physics| NULL|
| 13| Tariq Mehmood|Computer Science|   69|
| 14|    Sana Javed|     Mathematics|

## Step 4: Clean Missing Values

In [ ]:
from pyspark.sql.functions import count, when, isnan, isnull

print(' Missing Values BEFORE Cleaning:')
df.select([
    count(when(isnull(c) | (col(c).cast('string') == ''), c)).alias(c)
    for c in df.columns
]).show()


df_clean = df.filter(
    col('Marks').isNotNull() & (col('Marks').cast('string') != '')
)


df_clean = df_clean.withColumn('Marks', col('Marks').cast(IntegerType()))

print(' Missing Values AFTER Cleaning:')
df_clean.select([
    count(when(isnull(c), c)).alias(c)
    for c in df_clean.columns
]).show()

print(f'   Rows before cleaning : {df.count()}')
print(f'   Rows after  cleaning : {df_clean.count()}')
print(f'   Rows removed         : {df.count() - df_clean.count()}')

 Missing Values BEFORE Cleaning:
+---+----+----------+-----+
| ID|Name|Department|Marks|
+---+----+----------+-----+
|  0|   0|         0|    2|
+---+----+----------+-----+

 Missing Values AFTER Cleaning:
+---+----+----------+-----+
| ID|Name|Department|Marks|
+---+----+----------+-----+
|  0|   0|         0|    0|
+---+----+----------+-----+

   Rows before cleaning : 15
   Rows after  cleaning : 13
   Rows removed         : 2


## Step 5: Average Marks Per Department

In [13]:
avg_marks = df_clean.groupBy('Department') \
    .agg(spark_round(avg('Marks'), 2).alias('Average_Marks')) \
    .orderBy(desc('Average_Marks'))

print(' Average Marks Per Department:')
avg_marks.show()

 Average Marks Per Department:
+----------------+-------------+
|      Department|Average_Marks|
+----------------+-------------+
|     Mathematics|         86.0|
|         Physics|         84.5|
|Computer Science|         83.8|
+----------------+-------------+



## Step 6: Top 5 Students

In [14]:
top5 = df_clean.orderBy(desc('Marks')).limit(5)

print(' Top 5 Students:')
top5.show()

 Top 5 Students:
+---+--------------+----------------+-----+
| ID|          Name|      Department|Marks|
+---+--------------+----------------+-----+
|  7|    Hamza Raza|Computer Science|   95|
| 15|   Rizwan Shah|         Physics|   93|
|  2|     Sara Khan|     Mathematics|   92|
| 10|Nadia Siddiqui|Computer Science|   91|
|  5|   Bilal Ahmed|     Mathematics|   90|
+---+--------------+----------------+-----+



## Step 7: Summary Report

In [15]:
print('=' * 45)
print('       STUDENT PERFORMANCE SUMMARY        ')
print('=' * 45)
print(f'  Total Students (after cleaning) : {df_clean.count()}')
print()

print('  Average Marks by Department:')
for row in avg_marks.collect():
    print(f'     {row["Department"]:<20} : {row["Average_Marks"]}')

print()
print('  Top 5 Students:')
for i, row in enumerate(top5.collect(), 1):
    print(f'     {i}. {row["Name"]:<20} ({row["Department"]}) - {row["Marks"]} marks')

print('=' * 45)


spark.stop()
print('\n Spark Session Stopped.')

       STUDENT PERFORMANCE SUMMARY        
  Total Students (after cleaning) : 13

  Average Marks by Department:
     Mathematics          : 86.0
     Physics              : 84.5
     Computer Science     : 83.8

  Top 5 Students:
     1. Hamza Raza           (Computer Science) - 95 marks
     2. Rizwan Shah          (Physics) - 93 marks
     3. Sara Khan            (Mathematics) - 92 marks
     4. Nadia Siddiqui       (Computer Science) - 91 marks
     5. Bilal Ahmed          (Mathematics) - 90 marks

 Spark Session Stopped.
